# 10-coin benchmark comparison with multiple evaluation windows

This notebook benchmarks the 10-coin TDA indicators against external indicators available in `12.Data0329`.

Compared indicators:
- TDA indicators from the 24 `(method, m, tau)` groups
- Parkinson volatility for 10 assets
- Rogers volatility for 10 assets
- SP500
- VIX
- Fear & Greed index
- BVOL
- CVI

Evaluation windows:
- `[-21,0]`, `[-14,0]`, `[-7,0]`
- `[0,+7]`, `[0,+14]`, `[0,+21]`
- `[-7,+7]`, `[-14,+14]`, `[-21,+21]`

This notebook focuses on AUC-based benchmark comparison and stores all generated outputs in `12.Data0329/benchmark_multiwindow_10coin`.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc

TDA_DIR = Path('/Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_tda_csv')
MARKET_PATH = Path('/Users/jane/Documents/202511吾-Systems/12.Data0329/SP500_VIX_BVOL_CVI.csv')
FNG_PATH = Path('/Users/jane/Documents/202511吾-Systems/12.Data0329/fear_and_greed_index.csv')
PARK_PATH = Path('/Users/jane/Documents/202511吾-Systems/12.Data0329/Parkinson_volatility_10.csv')
ROGERS_PATH = Path('/Users/jane/Documents/202511吾-Systems/12.Data0329/Rogers_volatility_10.csv')
OUTPUT_DIR = Path('/Users/jane/Documents/202511吾-Systems/12.Data0329/benchmark_multiwindow_10coin')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tda_files = sorted([p for p in TDA_DIR.glob('close10_tda_*.csv') if p.name != 'close10_tda_manifest.csv'])
print(f'TDA files found: {len(tda_files)}')
print(f'Output directory: {OUTPUT_DIR}')


TDA files found: 24
Output directory: /Users/jane/Documents/202511吾-Systems/12.Data0329/benchmark_multiwindow_10coin


In [2]:
EVENTS = pd.to_datetime([
    '2020-12-01','2021-01-02','2021-01-07','2021-01-29','2021-02-16','2021-03-13',
    '2021-04-10','2021-05-12','2021-05-17','2021-05-18','2021-05-19','2021-06-09',
    '2021-09-24','2021-10-15','2021-11-15',
    '2022-04-27','2022-05-01','2022-05-11','2022-05-12','2022-05-13','2022-07-20',
    '2022-11-01','2023-03-01','2023-03-10','2023-05-17','2023-06-16','2023-07-01',
    '2023-10-01','2024-03-19','2024-04-20','2025-01-20','2025-02-03','2025-02-21',
    '2025-03-07','2025-05-20','2025-06-05','2025-06-17'
])

EVAL_WINDOWS = [
    ('[-21,0]', -21, 0),
    ('[-14,0]', -14, 0),
    ('[-7,0]', -7, 0),
    ('[0,+7]', 0, 7),
    ('[0,+14]', 0, 14),
    ('[0,+21]', 0, 21),
    ('[-7,+7]', -7, 7),
    ('[-14,+14]', -14, 14),
    ('[-21,+21]', -21, 21),
]

pd.DataFrame(EVAL_WINDOWS, columns=['window_label', 'start_offset', 'end_offset']).to_csv(
    OUTPUT_DIR / 'close10_evaluation_windows.csv', index=False
)
pd.DataFrame({'event_date': EVENTS}).to_csv(OUTPUT_DIR / 'close10_extreme_events.csv', index=False)

print('Saved evaluation windows and event dates.')


Saved evaluation windows and event dates.


In [3]:
tda_pattern = re.compile(r'close10_tda_(wasserstein|dtw)_m(\d+)_tau(\d+)\.csv')


def parse_tda_filename(path):
    m = tda_pattern.match(path.name)
    if not m:
        raise ValueError(f'Unexpected TDA filename: {path.name}')
    return m.group(1), int(m.group(2)), int(m.group(3))


tda_wide = None
indicator_manifest_rows = []

for path in tda_files:
    method, window_size, lag = parse_tda_filename(path)
    df = pd.read_csv(path, parse_dates=['date'])
    prefix = f'tda_{method}_m{window_size:02d}_tau{lag}'

    rename_map = {
        'betti0': f'{prefix}_betti0',
        'betti1': f'{prefix}_betti1',
        'persistent_entropy': f'{prefix}_entropy',
    }

    part = df[['date', 'betti0', 'betti1', 'persistent_entropy']].rename(columns=rename_map)

    if tda_wide is None:
        tda_wide = part.copy()
    else:
        tda_wide = tda_wide.merge(part, on='date', how='outer')

    for feature_name, indicator_col in [
        ('betti0', rename_map['betti0']),
        ('betti1', rename_map['betti1']),
        ('entropy', rename_map['persistent_entropy'])
    ]:
        indicator_manifest_rows.append({
            'indicator': indicator_col,
            'category': 'tda',
            'distance_method': method,
            'embedding_window_size': window_size,
            'embedding_lag': lag,
            'feature': feature_name,
            'benchmark_group': np.nan,
            'asset': np.nan,
            'source_file': path.name,
        })

tda_wide = tda_wide.sort_values('date').reset_index(drop=True)
tda_wide.to_csv(OUTPUT_DIR / 'close10_tda_wide_panel.csv', index=False)

indicator_manifest = pd.DataFrame(indicator_manifest_rows)
print('TDA wide shape:', tda_wide.shape)
indicator_manifest.head()


TDA wide shape: (1867, 73)


,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,benchmark_group,asset,source_file
0,tda_dtw_m07_tau1_betti0,tda,dtw,7,1,betti0,NaN,NaN,close10_tda_dtw_m07_tau1.csv
1,tda_dtw_m07_tau1_betti1,tda,dtw,7,1,betti1,NaN,NaN,close10_tda_dtw_m07_tau1.csv
2,tda_dtw_m07_tau1_entropy,tda,dtw,7,1,entropy,NaN,NaN,close10_tda_dtw_m07_tau1.csv
3,tda_dtw_m07_tau2_betti0,tda,dtw,7,2,betti0,NaN,NaN,close10_tda_dtw_m07_tau2.csv
4,tda_dtw_m07_tau2_betti1,tda,dtw,7,2,betti1,NaN,NaN,close10_tda_dtw_m07_tau2.csv


In [4]:
def ffill_internal_only(df, date_col, cols):
    df = df.copy()
    for col in cols:
        last_valid = df.loc[pd.to_numeric(df[col], errors='coerce').notna(), date_col].max()
        df[col] = pd.to_numeric(df[col], errors='coerce').ffill()
        if pd.notna(last_valid):
            df.loc[df[date_col] > last_valid, col] = np.nan
    return df


market_df = pd.read_csv(MARKET_PATH)
market_df['date'] = pd.to_datetime(market_df['Date'])
market_df = market_df[['date', 'SP500', 'VIX', 'BVOL', 'CVI']].sort_values('date').reset_index(drop=True)
market_df = ffill_internal_only(market_df, 'date', ['SP500', 'VIX', 'BVOL', 'CVI'])

fng_df = pd.read_csv(FNG_PATH)
fng_df['date'] = pd.to_datetime(fng_df['Date'])
fng_df = fng_df[['date', 'fng_value']].rename(columns={'fng_value': 'fear_greed'}).sort_values('date').reset_index(drop=True)
fng_df = ffill_internal_only(fng_df, 'date', ['fear_greed'])

park_df = pd.read_csv(PARK_PATH)
park_df['date'] = pd.to_datetime(park_df['Date'])
park_df = park_df.drop(columns=['Date']).sort_values('date').reset_index(drop=True)
park_assets = [c for c in park_df.columns if c != 'date']
park_rename = {c: f'parkinson_{c.lower()}' for c in park_assets}
park_df = park_df.rename(columns=park_rename)

rogers_df = pd.read_csv(ROGERS_PATH)
rogers_df['date'] = pd.to_datetime(rogers_df['Date'])
rogers_df = rogers_df.drop(columns=['Date']).sort_values('date').reset_index(drop=True)
rogers_assets = [c for c in rogers_df.columns if c != 'date']
rogers_rename = {c: f'rogers_{c.lower()}' for c in rogers_assets}
rogers_df = rogers_df.rename(columns=rogers_rename)

benchmark_panel = (
    tda_wide
    .merge(market_df, on='date', how='left')
    .merge(fng_df, on='date', how='left')
    .merge(park_df, on='date', how='left')
    .merge(rogers_df, on='date', how='left')
    .sort_values('date')
    .reset_index(drop=True)
)

benchmark_indicators = ['SP500', 'VIX', 'fear_greed', 'BVOL', 'CVI'] + list(park_rename.values()) + list(rogers_rename.values())

benchmark_rows = []
for col in ['SP500', 'VIX', 'fear_greed', 'BVOL', 'CVI']:
    benchmark_rows.append({
        'indicator': col,
        'category': 'benchmark',
        'distance_method': np.nan,
        'embedding_window_size': np.nan,
        'embedding_lag': np.nan,
        'feature': col,
        'benchmark_group': 'market',
        'asset': np.nan,
        'source_file': 'merged_benchmark_sources'
    })

for asset in park_assets:
    benchmark_rows.append({
        'indicator': f'parkinson_{asset.lower()}',
        'category': 'benchmark',
        'distance_method': np.nan,
        'embedding_window_size': np.nan,
        'embedding_lag': np.nan,
        'feature': 'parkinson',
        'benchmark_group': 'parkinson',
        'asset': asset,
        'source_file': Path(PARK_PATH).name
    })

for asset in rogers_assets:
    benchmark_rows.append({
        'indicator': f'rogers_{asset.lower()}',
        'category': 'benchmark',
        'distance_method': np.nan,
        'embedding_window_size': np.nan,
        'embedding_lag': np.nan,
        'feature': 'rogers',
        'benchmark_group': 'rogers',
        'asset': asset,
        'source_file': Path(ROGERS_PATH).name
    })

indicator_manifest = pd.concat([indicator_manifest, pd.DataFrame(benchmark_rows)], ignore_index=True)
indicator_manifest.to_csv(OUTPUT_DIR / 'close10_indicator_manifest.csv', index=False)
benchmark_panel.to_csv(OUTPUT_DIR / 'close10_benchmark_panel.csv', index=False)

coverage_rows = []
for col in benchmark_indicators:
    s = pd.to_numeric(benchmark_panel[col], errors='coerce')
    valid_dates = benchmark_panel.loc[s.notna(), 'date']
    coverage_rows.append({
        'indicator': col,
        'first_valid_date': valid_dates.min(),
        'last_valid_date': valid_dates.max(),
        'n_valid': int(s.notna().sum())
    })
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(OUTPUT_DIR / 'close10_benchmark_coverage.csv', index=False)

print('Benchmark panel shape:', benchmark_panel.shape)
print('Benchmark indicators:', len(benchmark_indicators))
benchmark_panel[['date'] + benchmark_indicators[:10]].head()


Benchmark panel shape: (1867, 98)
Benchmark indicators: 25


,date,SP500,VIX,fear_greed,BVOL,CVI,parkinson_eth,parkinson_btc,parkinson_xrp,parkinson_usdt,parkinson_usdc
0,2020-08-17,3381.989990,21.350000,83.0,40.26,85.7554,0.027886,0.027459,0.049655,0.011940,0.004937
1,2020-08-18,3389.780029,21.510000,82.0,40.50,85.8771,0.018191,0.018851,0.035843,0.007325,0.005899
2,2020-08-19,3374.850098,22.540001,80.0,42.96,81.9934,0.044271,0.017301,0.047141,0.006592,0.004333
3,2020-08-20,3385.510010,22.719999,75.0,43.01,76.9311,0.021475,0.009684,0.016607,0.002782,0.002516
4,2020-08-21,3397.159912,22.540001,81.0,43.15,75.7958,0.046509,0.017113,0.031841,0.007196,0.004026


In [5]:
for window_label, start_offset, end_offset in EVAL_WINDOWS:
    label_col = f'event_label_{window_label}'
    benchmark_panel[label_col] = 0
    for event_date in EVENTS:
        start_date = event_date + pd.Timedelta(days=start_offset)
        end_date = event_date + pd.Timedelta(days=end_offset)
        benchmark_panel.loc[(benchmark_panel['date'] >= start_date) & (benchmark_panel['date'] <= end_date), label_col] = 1

benchmark_panel.to_csv(OUTPUT_DIR / 'close10_benchmark_panel_with_labels.csv', index=False)
label_cols = [f'event_label_{w[0]}' for w in EVAL_WINDOWS]
benchmark_panel[['date'] + label_cols].head(20)


,date,"event_label_[-21,0]","event_label_[-14,0]","event_label_[-7,0]","event_label_[0,+7]","event_label_[0,+14]","event_label_[0,+21]","event_label_[-7,+7]","event_label_[-14,+14]","event_label_[-21,+21]"
0,2020-08-17,0,0,0,0,0,0,0,0,0
1,2020-08-18,0,0,0,0,0,0,0,0,0
2,2020-08-19,0,0,0,0,0,0,0,0,0
3,2020-08-20,0,0,0,0,0,0,0,0,0
4,2020-08-21,0,0,0,0,0,0,0,0,0
5,2020-08-22,0,0,0,0,0,0,0,0,0
6,2020-08-23,0,0,0,0,0,0,0,0,0
7,2020-08-24,0,0,0,0,0,0,0,0,0
8,2020-08-25,0,0,0,0,0,0,0,0,0
9,2020-08-26,0,0,0,0,0,0,0,0,0


In [6]:
tda_indicators = indicator_manifest.loc[indicator_manifest['category'] == 'tda', 'indicator'].tolist()
all_auc_indicators = tda_indicators + benchmark_indicators


def evaluable_event_count(series_dates, events, start_offset, end_offset):
    series_start = pd.to_datetime(series_dates.min())
    series_end = pd.to_datetime(series_dates.max())
    count = 0
    for e in events:
        ws = e + pd.Timedelta(days=start_offset)
        we = e + pd.Timedelta(days=end_offset)
        if ws >= series_start and we <= series_end:
            count += 1
    return count


auc_rows = []

for window_label, start_offset, end_offset in EVAL_WINDOWS:
    label_col = f'event_label_{window_label}'
    y_full = benchmark_panel[label_col].to_numpy()

    for indicator in all_auc_indicators:
        y_score = pd.to_numeric(benchmark_panel[indicator], errors='coerce').to_numpy()
        mask = np.isfinite(y_score)

        if mask.sum() == 0:
            continue

        y_true = y_full[mask]
        y_score_masked = y_score[mask]

        if np.unique(y_true).size < 2:
            continue

        fpr, tpr, _ = roc_curve(y_true, y_score_masked)
        roc_auc = auc(fpr, tpr)

        series_dates = benchmark_panel.loc[mask, 'date']
        meta = indicator_manifest.loc[indicator_manifest['indicator'] == indicator].iloc[0].to_dict()

        auc_rows.append({
            'window_label': window_label,
            'start_offset': start_offset,
            'end_offset': end_offset,
            'indicator': indicator,
            'category': meta['category'],
            'distance_method': meta['distance_method'],
            'embedding_window_size': meta['embedding_window_size'],
            'embedding_lag': meta['embedding_lag'],
            'feature': meta['feature'],
            'benchmark_group': meta['benchmark_group'],
            'asset': meta['asset'],
            'source_file': meta['source_file'],
            'auc': roc_auc,
            'n_obs': int(mask.sum()),
            'n_positive': int(y_true.sum()),
            'n_events_in_range': int(evaluable_event_count(series_dates, EVENTS, start_offset, end_offset)),
        })

auc_df = pd.DataFrame(auc_rows).sort_values(['window_label', 'auc'], ascending=[True, False]).reset_index(drop=True)
auc_df.to_csv(OUTPUT_DIR / 'close10_auc_multiwindow_long.csv', index=False)

pivot_meta_cols = ['indicator', 'category', 'distance_method', 'embedding_window_size', 'embedding_lag', 'feature', 'benchmark_group', 'asset']
auc_pivot_base = auc_df[pivot_meta_cols + ['window_label', 'auc']].copy()
for col in ['distance_method', 'feature', 'benchmark_group', 'asset']:
    auc_pivot_base[col] = auc_pivot_base[col].fillna('')
for col in ['embedding_window_size', 'embedding_lag']:
    auc_pivot_base[col] = auc_pivot_base[col].astype('Int64').astype(str).replace('<NA>', '')

auc_pivot = auc_pivot_base.pivot_table(
    index=pivot_meta_cols,
    columns='window_label',
    values='auc',
    aggfunc='first'
)
auc_pivot['mean_auc'] = auc_pivot.mean(axis=1)
auc_pivot = auc_pivot.sort_values('mean_auc', ascending=False).reset_index()
auc_pivot.to_csv(OUTPUT_DIR / 'close10_auc_multiwindow_pivot.csv', index=False)

auc_best = (
    auc_df.sort_values(['window_label', 'auc'], ascending=[True, False])
    .groupby('window_label', as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)
auc_best.to_csv(OUTPUT_DIR / 'close10_auc_best_by_window.csv', index=False)

indicator_summary = auc_df.groupby(
    ['indicator', 'category', 'distance_method', 'embedding_window_size', 'embedding_lag', 'feature', 'benchmark_group', 'asset'],
    dropna=False,
    as_index=False
)['auc'].mean().rename(columns={'auc': 'mean_auc'})
indicator_summary = indicator_summary.sort_values('mean_auc', ascending=False).reset_index(drop=True)
indicator_summary.to_csv(OUTPUT_DIR / 'close10_auc_indicator_summary.csv', index=False)

best_tda = (
    auc_df[auc_df['category'] == 'tda']
    .sort_values(['window_label', 'auc'], ascending=[True, False])
    .groupby('window_label', as_index=False)
    .first()
    .rename(columns={'indicator': 'best_tda_indicator', 'auc': 'best_tda_auc'})
)

best_benchmark = (
    auc_df[auc_df['category'] == 'benchmark']
    .sort_values(['window_label', 'auc'], ascending=[True, False])
    .groupby('window_label', as_index=False)
    .first()
    .rename(columns={'indicator': 'best_benchmark_indicator', 'auc': 'best_benchmark_auc', 'feature': 'best_benchmark_feature', 'benchmark_group': 'best_benchmark_group', 'asset': 'best_benchmark_asset'})
)

compare_df = best_tda[['window_label', 'best_tda_indicator', 'best_tda_auc']].merge(
    best_benchmark[['window_label', 'best_benchmark_indicator', 'best_benchmark_auc', 'best_benchmark_feature', 'best_benchmark_group', 'best_benchmark_asset']],
    on='window_label',
    how='outer'
)
compare_df['auc_gap_tda_minus_benchmark'] = compare_df['best_tda_auc'] - compare_df['best_benchmark_auc']
compare_df.to_csv(OUTPUT_DIR / 'close10_auc_best_tda_vs_benchmark_by_window.csv', index=False)

print('AUC result rows:', len(auc_df))
auc_df.head(10)


AUC result rows: 873


,window_label,start_offset,end_offset,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,benchmark_group,asset,source_file,auc,n_obs,n_positive,n_events_in_range
0,"[-14,+14]",-14,14,CVI,benchmark,NaN,NaN,NaN,CVI,market,NaN,merged_benchmark_sources,0.677459,1079,411,30
1,"[-14,+14]",-14,14,parkinson_trx,benchmark,NaN,NaN,NaN,parkinson,parkinson,TRX,Parkinson_volatility_10.csv,0.667349,1866,725,37
2,"[-14,+14]",-14,14,rogers_trx,benchmark,NaN,NaN,NaN,rogers,rogers,TRX,Rogers_volatility_10.csv,0.662287,1866,725,37
3,"[-14,+14]",-14,14,tda_wasserstein_m28_tau3_betti0,tda,wasserstein,28.0,3.0,betti0,NaN,NaN,close10_tda_wasserstein_m28_tau3.csv,0.661612,1792,725,37
4,"[-14,+14]",-14,14,tda_dtw_m28_tau3_betti0,tda,dtw,28.0,3.0,betti0,NaN,NaN,close10_tda_dtw_m28_tau3.csv,0.660732,1792,725,37
5,"[-14,+14]",-14,14,tda_dtw_m14_tau1_betti0,tda,dtw,14.0,1.0,betti0,NaN,NaN,close10_tda_dtw_m14_tau1.csv,0.656774,1860,725,37
6,"[-14,+14]",-14,14,tda_dtw_m28_tau1_betti0,tda,dtw,28.0,1.0,betti0,NaN,NaN,close10_tda_dtw_m28_tau1.csv,0.645618,1846,725,37
7,"[-14,+14]",-14,14,tda_dtw_m21_tau1_betti0,tda,dtw,21.0,1.0,betti0,NaN,NaN,close10_tda_dtw_m21_tau1.csv,0.645388,1853,725,37
8,"[-14,+14]",-14,14,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,NaN,NaN,close10_tda_dtw_m07_tau1.csv,0.644492,1867,725,37
9,"[-14,+14]",-14,14,tda_wasserstein_m21_tau3_betti0,tda,wasserstein,21.0,3.0,betti0,NaN,NaN,close10_tda_wasserstein_m21_tau3.csv,0.642598,1813,725,37


In [7]:
print('Saved files in output directory:')
for p in sorted(OUTPUT_DIR.glob('*')):
    print('-', p.name)

print('\nTop AUC rows:')
print(auc_df.head(10).to_string(index=False))

print('\nBest TDA vs benchmark by window:')
print(compare_df.to_string(index=False))


Saved files in output directory:
- close10_auc_best_by_window.csv
- close10_auc_best_tda_vs_benchmark_by_window.csv
- close10_auc_indicator_summary.csv
- close10_auc_multiwindow_long.csv
- close10_auc_multiwindow_pivot.csv
- close10_benchmark_coverage.csv
- close10_benchmark_panel.csv
- close10_benchmark_panel_with_labels.csv
- close10_evaluation_windows.csv
- close10_extreme_events.csv
- close10_indicator_manifest.csv
- close10_tda_wide_panel.csv

Top AUC rows:
window_label  start_offset  end_offset                       indicator  category distance_method  embedding_window_size  embedding_lag   feature benchmark_group asset                          source_file      auc  n_obs  n_positive  n_events_in_range
   [-14,+14]           -14          14                             CVI benchmark             NaN                    NaN            NaN       CVI          market   NaN             merged_benchmark_sources 0.677459   1079         411                 30
   [-14,+14]           -14     